In [7]:
import pandas as pd

# Load data
df = pd.read_csv("final.csv")

# إضافة User ID
df.insert(0, "user_id", range(1, len(df) + 1))

print(df.head())
# =========================
# Reduce size for analysis
# =========================

df_small = df.groupby('gender', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 100000), random_state=42)
)

print("Original size:", df.shape)
print("New size:", df_small.shape)

   user_id  age  gender  income  daily_gaming_hours  weekly_sessions  \
0        1   51  Female    8615                3.68               22   
1        2   41  Female   39453                5.70               34   
2        3   27    Male   40466                1.58                8   
3        4   55    Male   51076                6.11               39   
4        5   20    Male   86116                3.65               17   

   years_gaming  sleep_hours  caffeine_intake  exercise_hours  ...  \
0            17         5.26             1.00            0.18  ...   
1            16         9.20             0.70            1.44  ...   
2            22         7.39             2.24            3.15  ...   
3            24         7.99             1.65            2.80  ...   
4             0         7.12             1.02            1.01  ...   

   academic_level  most_used_platform  affects_academic_performance  \
0        Graduate            LinkedIn                            No   
1   

In [10]:
import pandas as pd
import numpy as np

# =========================
# 1. Start from sampled data
# =========================
df = df_small.copy()

# =========================
# 2. Standardize column names
# =========================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# =========================
# 3. Drop useless column
# =========================
if 'sleep_quality' in df.columns:
    df.drop(columns=['sleep_quality'], inplace=True)

# =========================
# 4. Remove duplicates
# =========================
df.drop_duplicates(inplace=True)

# =========================
# 5. Handle missing values
# =========================

# Numeric columns → median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical columns → mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# =========================
# 6. Fix text inconsistencies
# =========================
for col in cat_cols:
    df[col] = df[col].str.strip()

if 'gender' in df.columns:
    df['gender'] = df['gender'].str.capitalize()

# =========================
# 7. Optional: remove outliers (safe for analysis)
# =========================
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)

# =========================
# 8. Final check
# =========================
print("Final Shape:", df.shape)
print(df.info())
print("Missing values:", df.isnull().sum().sum())

# =========================
# 9. Save cleaned dataset
# =========================
df.to_csv("final_cleaned_sampleing.csv", index=False)

print(" Cleaned dataset saved successfully!")

C:\Users\Victus\AppData\Local\Temp\ipykernel_14000\3841452283.py:39: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


Final Shape: (239840, 48)
<class 'pandas.DataFrame'>
Index: 239840 entries, 173523 to 397363
Data columns (total 48 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   user_id                       239840 non-null  int64  
 1   age                           239840 non-null  int64  
 2   income                        239840 non-null  int64  
 3   daily_gaming_hours            239840 non-null  float64
 4   weekly_sessions               239840 non-null  int64  
 5   years_gaming                  239840 non-null  int64  
 6   sleep_hours                   239840 non-null  float64
 7   caffeine_intake               239840 non-null  float64
 8   exercise_hours                239840 non-null  float64
 9   stress_level                  239840 non-null  int64  
 10  anxiety_score                 239840 non-null  float64
 11  depression_score              239840 non-null  float64
 12  social_interaction_score     